# PEFT evidence on Kaggle (fallback for the HPC)
Same scripts as `evidence/` (HPC version), run on Kaggle's free T4 GPU.

**One-time:** verify your phone number (Kaggle → Settings), otherwise GPU and Internet stay locked.

1. Right panel → **Session options**: Accelerator **GPU T4 x2**, Internet **On**.
2. Quick test: run the first cells interactively until H1 starts printing `step 1/300 ...`, then stop it.
3. Real run: **Save Version → Save & Run All (Commit)**. It runs in the background (laptop can sleep), takes ~35–45 min for H1, ~1.5 h with all optional parts.
4. When the version finishes: open it → **Output** → download `results.zip`.

Runs use `--amp fp16` (T4 has no fast bf16); if a loss prints `nan`, set `AMP = "bf16"`. Finished runs are skipped when a cell is re-run.

In [ ]:
!nvidia-smi -L
%pip install -q "transformers>=4.56" peft "bitsandbytes>=0.45" datasets accelerate tabulate
# preinstalled torchao is too old for current PEFT and breaks plain LoRA; we don't use it
%pip uninstall -y -q torchao

In [ ]:
import os
WORK = '/kaggle/working/evidence'
os.environ['CKPT_ROOT'] = '/tmp/peft_ckpts'   # checkpoints off the saved output (keeps it small)
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

In [ ]:
%%writefile common.py
"""Shared helpers for the HPC evidence scripts (H1-H4)."""
from __future__ import annotations

import json
import os
import platform
import random
import socket
from datetime import datetime
from pathlib import Path

import numpy as np
import torch

EVIDENCE_DIR = Path(__file__).resolve().parent
RESULTS_DIR = Path(os.environ.get("RESULTS_DIR", EVIDENCE_DIR / "results"))
CKPT_ROOT = Path(os.environ.get("CKPT_ROOT", EVIDENCE_DIR / "checkpoints"))

ALPACA = "yahma/alpaca-cleaned"
SST2 = ("nyu-mll/glue", "sst2")
WIKITEXT = ("Salesforce/wikitext", "wikitext-2-raw-v1")
GIB = 1024 ** 3  # GPU memory is reported in GiB ("GB" on slides), matching how GPUs are labelled


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_amp_dtype(name: str) -> torch.dtype:
    """bf16 on Ampere+ (A100/A40/L40/H100), fp16 on older GPUs (V100/T4)."""
    if name == "auto":
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16
    return {"bf16": torch.bfloat16, "fp16": torch.float16}[name]


def dtype_name(dt: torch.dtype) -> str:
    return str(dt).replace("torch.", "")


def env_info() -> dict:
    info = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "hostname": socket.gethostname(),
        "slurm_job_id": os.environ.get("SLURM_JOB_ID"),
        "slurm_array_task_id": os.environ.get("SLURM_ARRAY_TASK_ID"),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
    }
    for pkg in ("transformers", "peft", "datasets", "accelerate", "bitsandbytes"):
        try:
            info[pkg] = __import__(pkg).__version__
        except Exception:
            info[pkg] = None
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info["gpu_name"] = p.name
        info["gpu_mem_gb"] = round(p.total_memory / GIB, 1)
    return info


def count_params(model) -> tuple[int, int]:
    """(trainable, total). PEFT's helper counts 4-bit packed weights correctly."""
    if hasattr(model, "get_nb_trainable_parameters"):
        return model.get_nb_trainable_parameters()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def lora_only_params(model) -> int:
    return sum(p.numel() for n, p in model.named_parameters() if "lora_" in n and p.requires_grad)


def reset_peak() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()


def mem_now_gb() -> float:
    return torch.cuda.memory_allocated() / GIB if torch.cuda.is_available() else 0.0


def peak_gb() -> tuple[float, float]:
    """(max allocated by tensors, max reserved by the caching allocator), in GB."""
    if not torch.cuda.is_available():
        return 0.0, 0.0
    return torch.cuda.max_memory_allocated() / GIB, torch.cuda.max_memory_reserved() / GIB


def weights_size_mb(path: Path) -> float:
    """Size of the weight files only (*.safetensors / *.bin), not tokenizer/config."""
    files = [f for f in Path(path).rglob("*") if f.is_file() and f.suffix in (".safetensors", ".bin")]
    return sum(f.stat().st_size for f in files) / 1e6


def is_oom(e: BaseException) -> bool:
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()


def write_result(exp: str, name: str, record: dict) -> Path:
    out = RESULTS_DIR / exp / f"{name}.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(record, indent=2, default=str))
    short = {k: v for k, v in record.items() if k not in ("env", "error_trace")}
    print("RESULT " + json.dumps(short, default=str), flush=True)
    print(f"[saved] {out}", flush=True)
    return out


In [ ]:
%%writefile train_llm.py
#!/usr/bin/env python
"""H1 / H2: full fine-tuning vs LoRA vs QLoRA on a causal LM (Alpaca-cleaned).

One invocation = one run = one JSON file in results/<exp>/<run>.json
(+ a per-step loss curve CSV next to it).

Out-of-memory is treated as a RESULT, not a crash: it is caught, recorded with the
stage where it happened, and the script exits 0 so the next method still runs.

Precision per method (same for H1 and H2):
  full  : fp32 master weights + bf16 autocast + AdamW  (classic mixed precision, ~16 bytes/param)
  lora  : frozen base in bf16, LoRA A/B in fp32, bf16 autocast
  qlora : frozen base in 4-bit NF4 (+ double quant), LoRA A/B in fp32, bf16 compute
"""
from __future__ import annotations

import argparse
import csv
import math
import sys
import time
import traceback
from pathlib import Path

import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          get_cosine_schedule_with_warmup)

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import (ALPACA, CKPT_ROOT, RESULTS_DIR, count_params, dtype_name, env_info,  # noqa: E402
                    is_oom, lora_only_params, mem_now_gb, peak_gb, pick_amp_dtype,
                    reset_peak, set_seed, weights_size_mb, write_result)

DATA_SEED = 1234  # fixed: every method/seed sees the same train/held-out split

PROMPT_INPUT = (
    "Below is an instruction that describes a task, paired with an input that provides further "
    "context. Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n")
PROMPT_NO_INPUT = (
    "Below is an instruction that describes a task. Write a response that appropriately "
    "completes the request.\n\n### Instruction:\n{instruction}\n\n### Response:\n")


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--method", choices=["full", "lora", "qlora"], required=True)
    p.add_argument("--model", default="Qwen/Qwen2.5-0.5B")
    p.add_argument("--exp", default="h1", help="results subfolder (h1 / h2)")
    p.add_argument("--run-name", default=None)
    p.add_argument("--steps", type=int, default=300, help="optimizer steps")
    p.add_argument("--n-examples", type=int, default=2000, help="total examples drawn from Alpaca")
    p.add_argument("--n-eval", type=int, default=200, help="held out from n-examples")
    p.add_argument("--max-len", type=int, default=512)
    p.add_argument("--micro-batch", type=int, default=4)
    p.add_argument("--grad-accum", type=int, default=4)
    p.add_argument("--lr", type=float, default=None, help="default: 1e-5 full, 2e-4 LoRA/QLoRA")
    p.add_argument("--warmup", type=int, default=10)
    p.add_argument("--lora-r", type=int, default=16)
    p.add_argument("--lora-alpha", type=int, default=32)
    p.add_argument("--lora-dropout", type=float, default=0.05)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--amp", choices=["auto", "bf16", "fp16"], default="auto")
    p.add_argument("--no-grad-ckpt", dest="grad_ckpt", action="store_false",
                   help="disable gradient checkpointing (on by default for ALL methods)")
    p.add_argument("--no-save", dest="save", action="store_false")
    p.add_argument("--skip-eval-before", action="store_true")
    p.add_argument("--log-every", type=int, default=10)
    p.add_argument("--timing-skip", type=int, default=5, help="first N steps excluded from speed")
    return p.parse_args()


# ----------------------------------------------------------------------------- data
def build_data(tok, args):
    ds = load_dataset(ALPACA, split="train").shuffle(seed=DATA_SEED).select(range(args.n_examples))

    def encode(ex):
        tmpl = PROMPT_INPUT if ex["input"].strip() else PROMPT_NO_INPUT
        p_ids = tok(tmpl.format(**ex), add_special_tokens=False)["input_ids"]
        r_ids = tok(ex["output"] + tok.eos_token, add_special_tokens=False)["input_ids"]
        ids = (p_ids + r_ids)[: args.max_len]
        labels = ([-100] * len(p_ids) + r_ids)[: args.max_len]   # loss on the response only
        return {"input_ids": ids, "labels": labels}

    ds = ds.map(encode, remove_columns=ds.column_names)
    ds = ds.filter(lambda ex: any(l != -100 for l in ex["labels"]))  # drop prompt-only truncations
    n_eval = min(args.n_eval, len(ds) // 5)
    train = ds.select(range(len(ds) - n_eval))
    held = ds.select(range(len(ds) - n_eval, len(ds)))
    return train, held


def collate(batch, pad_id):
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
    labels = torch.full((len(batch), L), -100, dtype=torch.long)
    attn = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"])
        labels[i, :n] = torch.tensor(b["labels"])
        attn[i, :n] = 1
    return {"input_ids": ids, "attention_mask": attn, "labels": labels}


def forever(loader):
    while True:
        yield from loader


# ----------------------------------------------------------------------------- model
def load_model(args, amp):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    gpu = {"": 0}
    if args.method == "full":
        model = AutoModelForCausalLM.from_pretrained(
            args.model, dtype=torch.float32, device_map=gpu)
    elif args.method == "lora":
        model = AutoModelForCausalLM.from_pretrained(args.model, dtype=amp, device_map=gpu)
    else:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=amp)
        model = AutoModelForCausalLM.from_pretrained(
            args.model, quantization_config=bnb, dtype=amp, device_map=gpu)
    model.config.use_cache = False

    gc_kwargs = {"use_reentrant": False}
    if args.method == "qlora":
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=args.grad_ckpt, gradient_checkpointing_kwargs=gc_kwargs)
    elif args.grad_ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gc_kwargs)

    if args.method in ("lora", "qlora"):
        if args.grad_ckpt:
            model.enable_input_require_grads()
        cfg = LoraConfig(r=args.lora_r, lora_alpha=args.lora_alpha, lora_dropout=args.lora_dropout,
                         target_modules="all-linear", bias="none", task_type="CAUSAL_LM")
        model = get_peft_model(model, cfg)
    return model


@torch.no_grad()
def evaluate(model, loader, amp):
    """Token-weighted mean cross-entropy on the held-out responses."""
    model.eval()
    total, n = 0.0, 0
    for b in loader:
        b = {k: v.cuda(non_blocking=True) for k, v in b.items()}
        with torch.autocast("cuda", dtype=amp):
            out = model(**b)
        k = int((b["labels"][:, 1:] != -100).sum())
        total += float(out.loss) * k
        n += k
    model.train()
    return total / max(n, 1)


# ----------------------------------------------------------------------------- main
def main():
    args = parse_args()
    set_seed(args.seed)
    amp = pick_amp_dtype(args.amp)
    lr = args.lr if args.lr is not None else (1e-5 if args.method == "full" else 2e-4)
    run = args.run_name or f"{args.method}_seed{args.seed}"
    is_peft = args.method != "full"

    rec = {
        "exp": args.exp, "run": run, "method": args.method, "model": args.model, "status": "started",
        "seed": args.seed, "steps": args.steps, "micro_batch": args.micro_batch,
        "grad_accum": args.grad_accum, "effective_batch": args.micro_batch * args.grad_accum,
        "max_len": args.max_len, "lr": lr, "amp_dtype": dtype_name(amp), "grad_ckpt": args.grad_ckpt,
        "lora_r": args.lora_r if is_peft else None, "lora_alpha": args.lora_alpha if is_peft else None,
        "lora_targets": "all-linear" if is_peft else None, "env": env_info(),
    }
    print(f"=== {args.exp}/{run}: {args.method} on {args.model} | amp={rec['amp_dtype']} "
          f"| GPU={rec['env'].get('gpu_name')} ({rec['env'].get('gpu_mem_gb')} GB)", flush=True)

    t_wall = time.time()
    stage, steps_done, curve = "load_data", 0, []
    try:
        tok = AutoTokenizer.from_pretrained(args.model)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        train_ds, held_ds = build_data(tok, args)
        rec.update(n_train=len(train_ds), n_eval=len(held_ds))

        stage = "load_model"
        reset_peak()
        model = load_model(args, amp)
        rec["mem_after_load_gb"] = round(mem_now_gb(), 3)
        trainable, total = count_params(model)
        rec.update(trainable_params=trainable, total_params=total,
                   trainable_pct=round(100 * trainable / total, 4),
                   lora_params=lora_only_params(model) if is_peft else None)
        if is_peft:
            model.print_trainable_parameters()
        print(f"trainable={trainable:,} total={total:,} ({rec['trainable_pct']}%) "
              f"| weights on GPU={rec['mem_after_load_gb']} GB", flush=True)

        g = torch.Generator().manual_seed(args.seed)
        coll = lambda b: collate(b, tok.pad_token_id)  # noqa: E731
        train_loader = DataLoader(train_ds, batch_size=args.micro_batch, shuffle=True,
                                  generator=g, collate_fn=coll, num_workers=2, pin_memory=True)
        held_loader = DataLoader(held_ds, batch_size=args.micro_batch, shuffle=False,
                                 collate_fn=coll, num_workers=2)

        if not args.skip_eval_before:
            stage = "eval_before"
            rec["eval_loss_before"] = round(evaluate(model, held_loader, amp), 4)
            print(f"held-out loss before training: {rec['eval_loss_before']}", flush=True)

        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
        sched = get_cosine_schedule_with_warmup(opt, args.warmup, args.steps)
        scaler = torch.amp.GradScaler("cuda", enabled=(amp == torch.float16))

        model.train()
        it = forever(train_loader)
        reset_peak()  # peak from here on = weights + grads + optimizer states + activations
        timed_s, timed_tok = 0.0, 0
        for step in range(args.steps):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            stage = f"train_step{step}_forward_backward"
            step_loss, step_tok = 0.0, 0
            for _ in range(args.grad_accum):
                b = {k: v.cuda(non_blocking=True) for k, v in next(it).items()}
                with torch.autocast("cuda", dtype=amp):
                    loss = model(**b).loss / args.grad_accum
                scaler.scale(loss).backward()
                step_loss += float(loss)
                step_tok += int(b["attention_mask"].sum())
            stage = f"train_step{step}_optimizer"
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            sched.step()
            torch.cuda.synchronize()
            dt = time.perf_counter() - t0
            steps_done = step + 1
            if step >= args.timing_skip:
                timed_s += dt
                timed_tok += step_tok
            curve.append((steps_done, round(step_loss, 5), round(dt, 4)))
            if steps_done % args.log_every == 0 or steps_done == 1:
                pa, _ = peak_gb()
                print(f"step {steps_done:4d}/{args.steps} loss {step_loss:.4f} {dt:.3f}s/step "
                      f"peak {pa:.2f} GB", flush=True)

        pa, pr = peak_gb()
        n_timed = max(steps_done - args.timing_skip, 1)
        rec.update(peak_mem_gb=round(pa, 3), peak_reserved_gb=round(pr, 3),
                   sec_per_step=round(timed_s / n_timed, 4),
                   tokens_per_sec=round(timed_tok / timed_s, 1) if timed_s else None,
                   train_loss_last20=round(sum(c[1] for c in curve[-20:]) / len(curve[-20:]), 4),
                   steps_done=steps_done)

        stage = "eval_after"
        rec["eval_loss_after"] = round(evaluate(model, held_loader, amp), 4)
        rec["eval_ppl_after"] = round(math.exp(rec["eval_loss_after"]), 3)
        print(f"held-out loss after training: {rec['eval_loss_after']}", flush=True)

        if args.save:
            stage = "save"
            out_dir = CKPT_ROOT / args.exp / run
            out_dir.mkdir(parents=True, exist_ok=True)
            del opt
            if args.method == "full":
                model = model.to(torch.bfloat16)  # ship in 16-bit, like any released model
                model.save_pretrained(out_dir, safe_serialization=True)
                tok.save_pretrained(out_dir)
            else:
                model.save_pretrained(out_dir)  # adapter weights only
            rec["ckpt_dir"] = str(out_dir)
            rec["ckpt_size_mb"] = round(weights_size_mb(out_dir), 1)
        rec["status"] = "ok"
        exit_code = 0
    except Exception as e:  # noqa: BLE001
        pa, pr = peak_gb()
        rec.update(peak_mem_gb=round(pa, 3), peak_reserved_gb=round(pr, 3), steps_done=steps_done,
                   failed_stage=stage, error=str(e).splitlines()[0][:300],
                   error_trace=traceback.format_exc())
        traceback.print_exc()
        if is_oom(e):
            rec["status"] = "OOM"
            print(f"\n*** OUT OF MEMORY at stage '{stage}' after {steps_done} steps; "
                  f"peak allocated {pa:.2f} GB of {rec['env'].get('gpu_mem_gb')} GB ***\n", flush=True)
            exit_code = 0  # OOM is a measured outcome (H2)
        else:
            rec["status"] = "error"
            exit_code = 1

    rec["wall_time_s"] = round(time.time() - t_wall, 1)
    if curve:
        cpath = RESULTS_DIR / args.exp / f"{run}_curve.csv"
        cpath.parent.mkdir(parents=True, exist_ok=True)
        with open(cpath, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["step", "train_loss", "sec"])
            w.writerows(curve)
    write_result(args.exp, run, rec)
    sys.exit(exit_code)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_h3.py
#!/usr/bin/env python
"""H3: LoRA rank sweep on DistilBERT + SST-2 (plus a full fine-tuning reference).

One invocation = one (method, rank, seed) run = results/h3/<run>.json
LoRA targets q_lin + v_lin (as in the LoRA paper); the classification head
(pre_classifier + classifier, randomly initialised) is trained in every run.
By default alpha = 2*r so the LoRA scale alpha/r stays constant across the sweep.
"""
from __future__ import annotations

import argparse
import sys
import time
import traceback
from pathlib import Path

import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          get_linear_schedule_with_warmup)

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import (CKPT_ROOT, SST2, count_params, dtype_name, env_info, lora_only_params,  # noqa: E402
                    peak_gb, pick_amp_dtype, reset_peak, set_seed, weights_size_mb, write_result)


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--method", choices=["full", "lora"], default="lora")
    p.add_argument("--r", type=int, default=8)
    p.add_argument("--alpha", type=int, default=None, help="default 2*r (constant scale)")
    p.add_argument("--dropout", type=float, default=0.1)
    p.add_argument("--targets", default="q_lin,v_lin")
    p.add_argument("--model", default="distilbert/distilbert-base-uncased")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--epochs", type=int, default=2)
    p.add_argument("--batch", type=int, default=32)
    p.add_argument("--max-len", type=int, default=128)
    p.add_argument("--lr", type=float, default=None, help="default 2e-5 full, 5e-4 LoRA")
    p.add_argument("--train-subset", type=int, default=0, help="0 = full 67k train set")
    p.add_argument("--amp", choices=["auto", "bf16", "fp16"], default="auto")
    p.add_argument("--exp", default="h3")
    return p.parse_args()


@torch.no_grad()
def accuracy(model, loader, amp):
    model.eval()
    correct = n = 0
    for b in loader:
        b = {k: v.cuda() for k, v in b.items()}
        with torch.autocast("cuda", dtype=amp):
            logits = model(**{k: v for k, v in b.items() if k != "labels"}).logits
        correct += int((logits.argmax(-1) == b["labels"]).sum())
        n += len(b["labels"])
    model.train()
    return correct / n


def main():
    args = parse_args()
    set_seed(args.seed)
    amp = pick_amp_dtype(args.amp)
    is_lora = args.method == "lora"
    alpha = (args.alpha or 2 * args.r) if is_lora else None
    lr = args.lr if args.lr is not None else (5e-4 if is_lora else 2e-5)
    run = f"lora_r{args.r}_seed{args.seed}" if is_lora else f"full_seed{args.seed}"
    rec = {"exp": args.exp, "run": run, "method": args.method, "model": args.model,
           "lora_r": args.r if is_lora else None, "lora_alpha": alpha,
           "lora_targets": args.targets if is_lora else None, "seed": args.seed, "epochs": args.epochs,
           "batch": args.batch, "max_len": args.max_len, "lr": lr, "amp_dtype": dtype_name(amp),
           "status": "started", "env": env_info()}
    print(f"=== h3/{run} | GPU={rec['env'].get('gpu_name')}", flush=True)
    t_wall = time.time()
    try:
        tok = AutoTokenizer.from_pretrained(args.model)
        ds = load_dataset(*SST2)
        train = ds["train"]
        if args.train_subset:
            train = train.shuffle(seed=1234).select(range(args.train_subset))
        val = ds["validation"]  # 872 labelled examples (test labels are hidden)

        def enc(batch):
            return tok(batch["sentence"], truncation=True, max_length=args.max_len)
        cols = ["sentence", "idx"]
        train = train.map(enc, batched=True, remove_columns=cols).rename_column("label", "labels")
        val = val.map(enc, batched=True, remove_columns=cols).rename_column("label", "labels")
        coll = DataCollatorWithPadding(tok)
        g = torch.Generator().manual_seed(args.seed)
        train_loader = DataLoader(train, batch_size=args.batch, shuffle=True, generator=g,
                                  collate_fn=coll, num_workers=2)
        val_loader = DataLoader(val, batch_size=128, collate_fn=coll)
        rec.update(n_train=len(train), n_val=len(val))

        reset_peak()
        model = AutoModelForSequenceClassification.from_pretrained(
            args.model, num_labels=2, dtype=torch.float32).cuda()  # fp32 master weights, bf16 autocast
        if is_lora:
            from peft import LoraConfig, get_peft_model
            from peft.tuners.lora import LoraLayer
            cfg = LoraConfig(task_type="SEQ_CLS", r=args.r, lora_alpha=alpha, lora_dropout=args.dropout,
                             target_modules=args.targets.split(","),
                             modules_to_save=["pre_classifier", "classifier"])
            model = get_peft_model(model, cfg)
            model.print_trainable_parameters()
            # Paper formula check: r*(d_in + d_out) per adapted matrix
            analytic = sum(args.r * (m.get_base_layer().in_features + m.get_base_layer().out_features)
                           for m in model.modules() if isinstance(m, LoraLayer))
            n_mat = sum(1 for m in model.modules() if isinstance(m, LoraLayer))
            measured = lora_only_params(model)
            rec.update(lora_params=measured, lora_params_analytic=analytic, lora_matrices=n_mat,
                       analytic_match=(measured == analytic))
            print(f"LoRA params measured={measured:,} analytic r*(d+k) over {n_mat} matrices={analytic:,} "
                  f"match={measured == analytic}", flush=True)
        trainable, total = count_params(model)
        rec.update(trainable_params=trainable, total_params=total,
                   trainable_pct=round(100 * trainable / total, 4),
                   head_params=(trainable - rec.get("lora_params", 0)) if is_lora else None)

        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        total_steps = args.epochs * len(train_loader)
        sched = get_linear_schedule_with_warmup(opt, int(0.06 * total_steps), total_steps)
        scaler = torch.amp.GradScaler("cuda", enabled=(amp == torch.float16))

        reset_peak()
        epoch_times, epoch_acc = [], []
        model.train()
        for ep in range(args.epochs):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            for i, b in enumerate(train_loader):
                b = {k: v.cuda(non_blocking=True) for k, v in b.items()}
                with torch.autocast("cuda", dtype=amp):
                    loss = model(**b).loss
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
                if i % 500 == 0:
                    print(f"epoch {ep + 1} step {i}/{len(train_loader)} loss {float(loss):.4f}", flush=True)
            torch.cuda.synchronize()
            epoch_times.append(round(time.perf_counter() - t0, 2))
            epoch_acc.append(round(accuracy(model, val_loader, amp), 4))
            print(f"epoch {ep + 1}: {epoch_times[-1]}s  val_acc={epoch_acc[-1]}", flush=True)

        pa, pr = peak_gb()
        out_dir = CKPT_ROOT / args.exp / run
        model.save_pretrained(out_dir)
        rec.update(val_acc=epoch_acc[-1], val_acc_per_epoch=epoch_acc,
                   sec_per_epoch=round(sum(epoch_times) / len(epoch_times), 2),
                   peak_mem_gb=round(pa, 3), peak_reserved_gb=round(pr, 3),
                   ckpt_size_mb=round(weights_size_mb(out_dir), 2), status="ok")
        code = 0
    except Exception as e:  # noqa: BLE001
        traceback.print_exc()
        rec.update(status="error", error=str(e).splitlines()[0][:300], error_trace=traceback.format_exc())
        code = 1
    rec["wall_time_s"] = round(time.time() - t_wall, 1)
    write_result(args.exp, run, rec)
    sys.exit(code)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval_h4.py
#!/usr/bin/env python
"""H4: "forgets less?" Perplexity on WikiText-2 (test) for the base model and each H1 model.

Variants (each -> results/h4/<variant>.json):
  base      H1 base model in bf16                 (reference for full + lora)
  base4bit  H1 base model in 4-bit NF4            (reference for qlora: isolates the quantization effect)
  full      H1 full fine-tune checkpoint
  lora      base bf16 + H1 LoRA adapter (merged)
  qlora     base 4-bit + H1 QLoRA adapter
Sliding-window perplexity (window 1024, stride 512), the standard Hugging Face recipe.
"""
from __future__ import annotations

import argparse
import gc
import math
import sys
import time
import traceback
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import (CKPT_ROOT, WIKITEXT, dtype_name, env_info, peak_gb, pick_amp_dtype,  # noqa: E402
                    reset_peak, write_result)

ALL = ["base", "base4bit", "full", "lora", "qlora"]


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--base", default="Qwen/Qwen2.5-0.5B", help="must be the model H1 trained")
    p.add_argument("--variants", nargs="+", default=ALL, choices=ALL)
    p.add_argument("--h1-dir", default=str(CKPT_ROOT / "h1"), help="where H1 saved its checkpoints")
    p.add_argument("--seed", type=int, default=42, help="which H1 seed's checkpoints to evaluate")
    p.add_argument("--window", type=int, default=1024)
    p.add_argument("--stride", type=int, default=512)
    p.add_argument("--max-tokens", type=int, default=0, help="0 = whole test set")
    p.add_argument("--amp", choices=["auto", "bf16", "fp16"], default="auto")
    p.add_argument("--exp", default="h4")
    return p.parse_args()


def load_variant(v, args, amp):
    from peft import PeftModel
    gpu = {"": 0}
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=amp)
    ck = Path(args.h1_dir)
    if v == "base":
        return AutoModelForCausalLM.from_pretrained(args.base, dtype=amp, device_map=gpu), args.base
    if v == "base4bit":
        return AutoModelForCausalLM.from_pretrained(args.base, quantization_config=bnb, dtype=amp,
                                                    device_map=gpu), args.base + " (nf4)"
    path = ck / f"{v}_seed{args.seed}"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found - did H1 finish with saving enabled?")
    if v == "full":
        return AutoModelForCausalLM.from_pretrained(path, dtype=amp, device_map=gpu), str(path)
    if v == "lora":
        m = AutoModelForCausalLM.from_pretrained(args.base, dtype=amp, device_map=gpu)
        return PeftModel.from_pretrained(m, path).merge_and_unload(), str(path)
    m = AutoModelForCausalLM.from_pretrained(args.base, quantization_config=bnb, dtype=amp, device_map=gpu)
    return PeftModel.from_pretrained(m, path), str(path)


@torch.no_grad()
def perplexity(model, ids, window, stride, amp):
    model.eval()
    seq_len = ids.size(1)
    nll_sum, n_tok, prev_end = 0.0, 0, 0
    for begin in range(0, seq_len, stride):
        end = min(begin + window, seq_len)
        trg = end - prev_end  # only score tokens not scored by the previous window
        x = ids[:, begin:end].cuda()
        y = x.clone()
        y[:, :-trg] = -100
        with torch.autocast("cuda", dtype=amp):
            loss = model(input_ids=x, labels=y).loss
        k = int((y[:, 1:] != -100).sum())
        nll_sum += float(loss) * k
        n_tok += k
        prev_end = end
        if end == seq_len:
            break
    return math.exp(nll_sum / n_tok), nll_sum / n_tok, n_tok


def main():
    args = parse_args()
    amp = pick_amp_dtype(args.amp)
    tok = AutoTokenizer.from_pretrained(args.base)
    test = load_dataset(*WIKITEXT, split="test")
    ids = tok("\n\n".join(test["text"]), return_tensors="pt").input_ids
    if args.max_tokens:
        ids = ids[:, : args.max_tokens]
    print(f"WikiText-2 test: {ids.size(1):,} tokens", flush=True)

    failures = 0
    for v in args.variants:
        rec = {"exp": args.exp, "run": v, "variant": v, "base_model": args.base, "window": args.window,
               "stride": args.stride, "amp_dtype": dtype_name(amp), "h1_seed": args.seed, "env": env_info()}
        t0 = time.time()
        model = None
        try:
            reset_peak()
            model, src = load_variant(v, args, amp)
            ppl, nll, n = perplexity(model, ids, args.window, args.stride, amp)
            pa, _ = peak_gb()
            rec.update(source=src, wikitext2_ppl=round(ppl, 4), wikitext2_nll=round(nll, 5),
                       n_tokens_scored=n, peak_mem_gb=round(pa, 3), status="ok")
            print(f"[{v}] WikiText-2 perplexity = {ppl:.3f}", flush=True)
        except Exception as e:  # noqa: BLE001
            traceback.print_exc()
            failures += 1
            rec.update(status="error", error=str(e).splitlines()[0][:300])
        rec["wall_time_s"] = round(time.time() - t0, 1)
        write_result(args.exp, v, rec)
        del model
        gc.collect()
        torch.cuda.empty_cache()
    sys.exit(1 if failures else 0)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile make_report.py
#!/usr/bin/env python
"""Collect results/h*/ JSON files into slide-ready tables (Markdown + CSV) and PNG charts.

Runs on CPU (login node or laptop):  python make_report.py
Output: results/summary/  (h1_table.md, h1_memory.png, ..., footer.txt)
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parent))
from common import RESULTS_DIR  # noqa: E402

OUT = RESULTS_DIR / "summary"
# Fixed identity colours (same method = same colour on every chart)
COLOR = {"full": "#2a78d6", "lora": "#eb6834", "qlora": "#1baf7a"}
LABEL = {"full": "Full FT", "lora": "LoRA", "qlora": "QLoRA"}
INK, MUTED, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"
ORDER = ["full", "lora", "qlora"]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e4e3df",
    "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 11, "axes.titlesize": 13,
})


def load(exp: str) -> pd.DataFrame:
    rows = []
    for f in sorted((RESULTS_DIR / exp).glob("*.json")):
        r = json.loads(f.read_text())
        env = r.pop("env", {}) or {}
        r.pop("error_trace", None)
        rows.append({**r, **{f"env_{k}": v for k, v in env.items()}})
    return pd.DataFrame(rows)


def fmt_params(n):
    if pd.isna(n):
        return "-"
    n = float(n)
    return f"{n / 1e9:.2f}B" if n >= 1e9 else f"{n / 1e6:.1f}M" if n >= 1e6 else f"{n / 1e3:.1f}K"


def fmt_size(mb):
    if pd.isna(mb):
        return "-"
    return f"{mb / 1e3:.2f} GB" if mb >= 1000 else f"{mb:.1f} MB"


def num(x, d=2):
    return "-" if x is None or pd.isna(x) else f"{x:.{d}f}"


def save_table(df: pd.DataFrame, name: str):
    OUT.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT / f"{name}.csv", index=False)
    (OUT / f"{name}.md").write_text(df.to_markdown(index=False))
    print(f"\n## {name}\n{df.to_markdown(index=False)}")


def bar(ax, labels, values, colors, texts):
    xs = range(len(labels))
    ax.bar(xs, values, color=colors, width=0.55, edgecolor=SURFACE, linewidth=2)
    for x, v, t in zip(xs, values, texts):
        ax.annotate(t, (x, v), xytext=(0, 4), textcoords="offset points", ha="center", va="bottom",
                    color=INK, fontsize=10)
    ax.set_xticks(list(xs), labels)
    ax.grid(axis="x", visible=False)


def gpu_line(df):
    return f"1x {df['env_gpu_name'].dropna().iloc[0]} ({df['env_gpu_mem_gb'].dropna().iloc[0]} GB)" \
        if "env_gpu_name" in df and df["env_gpu_name"].notna().any() else "GPU ?"


# ----------------------------------------------------------------------------- H1
def report_h1():
    df = load("h1")
    if df.empty:
        return print("H1: no results yet")
    df = df[df.status == "ok"]
    agg = df.groupby("method").agg(
        trainable=("trainable_params", "first"), total=("total_params", "first"),
        pct=("trainable_pct", "first"), mem=("peak_mem_gb", "mean"), sps=("sec_per_step", "mean"),
        tps=("tokens_per_sec", "mean"), before=("eval_loss_before", "mean"),
        after=("eval_loss_after", "mean"), size=("ckpt_size_mb", "mean"), n=("seed", "count"),
    ).reindex([m for m in ORDER if m in set(df.method)])
    table = pd.DataFrame({
        "Method": [LABEL[m] for m in agg.index],
        "Trainable params": [fmt_params(v) for v in agg.trainable],
        "% of total": [f"{v:.2f}%" for v in agg.pct],
        "Peak GPU mem (GB)": [num(v) for v in agg.mem],
        "Sec / step": [num(v, 3) for v in agg.sps],
        "Tokens / s": [num(v, 0) for v in agg.tps],
        "Held-out loss (before -> after)": [f"{num(b, 3)} -> {num(a, 3)}" for b, a in zip(agg.before, agg.after)],
        "Checkpoint size": [fmt_size(v) for v in agg["size"]],
        "Seeds": agg.n.tolist(),
    })
    save_table(table, "h1_table")

    # derived ratios for the talk track
    if {"full", "lora"} <= set(agg.index):
        lines = [f"LoRA trains {agg.trainable['full'] / agg.trainable['lora']:.0f}x fewer params than full FT",
                 f"LoRA peak memory = {agg.mem['lora'] / agg.mem['full']:.2f}x of full FT "
                 f"({agg.mem['full'] / agg.mem['lora']:.1f}x less)",
                 f"LoRA checkpoint = {agg['size']['full'] / agg['size']['lora']:.0f}x smaller"]
        if "qlora" in agg.index:
            lines += [f"QLoRA peak memory = {agg.mem['full'] / agg.mem['qlora']:.1f}x less than full FT, "
                      f"{agg.mem['lora'] / agg.mem['qlora']:.2f}x less than LoRA",
                      f"QLoRA sec/step = {agg.sps['qlora'] / agg.sps['lora']:.2f}x LoRA's (>1 means slower)"]
        (OUT / "h1_ratios.txt").write_text("\n".join(lines) + "\n")
        print("\n".join(lines))

    ms = list(agg.index)
    cols = [COLOR[m] for m in ms]
    fig, ax = plt.subplots(figsize=(6.4, 4))
    bar(ax, [LABEL[m] for m in ms], agg.mem, cols, [f"{v:.1f} GB\nloss {a:.3f}" for v, a in zip(agg.mem, agg.after)])
    ax.set_ylabel("Peak GPU memory (GB)")
    ax.set_title(f"H1 - {df.model.iloc[0].split('/')[-1]}, {gpu_line(df)}", loc="left")
    ax.set_ylim(0, agg.mem.max() * 1.25)
    fig.tight_layout(); fig.savefig(OUT / "h1_memory.png", dpi=200); plt.close(fig)

    fig, ax = plt.subplots(figsize=(6.4, 4))
    bar(ax, [LABEL[m] for m in ms], agg.sps, cols, [f"{v:.2f} s" for v in agg.sps])
    ax.set_ylabel("Seconds per optimizer step (lower is faster)")
    ax.set_title("H1 - training speed", loc="left")
    ax.set_ylim(0, agg.sps.max() * 1.2)
    fig.tight_layout(); fig.savefig(OUT / "h1_speed.png", dpi=200); plt.close(fig)

    fig, ax = plt.subplots(figsize=(6.4, 4))
    for m in ms:
        seed = df[df.method == m].seed.iloc[0]
        c = RESULTS_DIR / "h1" / f"{m}_seed{seed}_curve.csv"
        if c.exists():
            cur = pd.read_csv(c)
            ax.plot(cur.step, cur.train_loss.rolling(10, min_periods=1).mean(), color=COLOR[m], lw=2, label=LABEL[m])
    ax.set_xlabel("Optimizer step"); ax.set_ylabel("Train loss (10-step mean)")
    ax.set_title("H1 - training loss", loc="left"); ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(OUT / "h1_loss_curves.png", dpi=200); plt.close(fig)
    return df


# ----------------------------------------------------------------------------- H2
def report_h2():
    df = load("h2")
    if df.empty:
        return print("H2: no results yet")
    model_name = df.model.iloc[0].split("/")[-1]
    df = df.set_index("method").reindex([m for m in ORDER if m in set(df.method)])
    gpu_gb = df["env_gpu_mem_gb"].dropna().iloc[0]
    def outcome(r):
        if r.status == "OOM":
            return f"OOM at {str(r.failed_stage).replace('_', ' ')}"
        return "trains" if r.status == "ok" else r.status
    table = pd.DataFrame({
        f"Method ({model_name})": [LABEL[m] for m in df.index],
        "Outcome": [outcome(r) for _, r in df.iterrows()],
        "Trainable params": [fmt_params(v) for v in df.get("trainable_params", pd.Series(index=df.index))],
        "Weights on GPU (GB)": [num(v) for v in df.get("mem_after_load_gb", pd.Series(index=df.index))],
        "Peak GPU mem (GB)": [num(v) if r.status == "ok" else f"OOM (> {gpu_gb:.0f} GB)" for v, (_, r)
                              in zip(df.peak_mem_gb, df.iterrows())],
        "Sec / step": [num(v, 2) for v in df.get("sec_per_step", pd.Series(index=df.index))],
    })
    save_table(table, "h2_table")

    fig, ax = plt.subplots(figsize=(6.4, 4))
    ms = list(df.index)
    vals = [gpu_gb if df.loc[m, "status"] == "OOM" else df.loc[m, "peak_mem_gb"] for m in ms]
    bars = ax.bar(range(len(ms)), vals, color=[COLOR[m] for m in ms], width=0.55, edgecolor=SURFACE, linewidth=2)
    for b, m, v in zip(bars, ms, vals):
        oom = df.loc[m, "status"] == "OOM"
        if oom:
            b.set_hatch("//"); b.set_alpha(0.45)
        ax.annotate("OOM" if oom else f"{v:.1f} GB", (b.get_x() + b.get_width() / 2, v), xytext=(0, 4),
                    textcoords="offset points", ha="center", color=INK, fontsize=10)
    ax.axhline(gpu_gb, color=MUTED, ls="--", lw=1)
    ax.annotate(f"GPU capacity {gpu_gb:.0f} GB", (len(ms) - 1, gpu_gb), xytext=(0, 4),
                textcoords="offset points", ha="center", color=MUTED, fontsize=9)
    ax.set_xticks(range(len(ms)), [LABEL[m] for m in ms]); ax.grid(axis="x", visible=False)
    ax.set_ylabel("Peak GPU memory (GB)"); ax.set_ylim(0, gpu_gb * 1.15)
    ax.set_title(f"H2 - {model_name} on {gpu_line(df.reset_index())}", loc="left")
    fig.tight_layout(); fig.savefig(OUT / "h2_memory.png", dpi=200); plt.close(fig)
    return df.reset_index()


# ----------------------------------------------------------------------------- H3
def report_h3():
    df = load("h3")
    if df.empty:
        return print("H3: no results yet")
    df = df[df.status == "ok"]
    lo = df[df.method == "lora"].groupby("lora_r").agg(
        acc_mean=("val_acc", "mean"), acc_min=("val_acc", "min"), acc_max=("val_acc", "max"),
        lora=("lora_params", "first"), analytic=("lora_params_analytic", "first"),
        match=("analytic_match", "all"), trainable=("trainable_params", "first"),
        pct=("trainable_pct", "first"), size=("ckpt_size_mb", "first"), n=("seed", "count")).sort_index()
    full = df[df.method == "full"]
    table = pd.DataFrame({
        "Rank r": [int(r) for r in lo.index],
        "LoRA params (measured)": [f"{int(v):,}" for v in lo.lora],
        "r(d+k) analytic": [f"{int(v):,}" for v in lo.analytic],
        "Match": ["yes" if m else "NO" for m in lo.match],
        "Trainable incl. head": [f"{int(v):,} ({p:.2f}%)" for v, p in zip(lo.trainable, lo.pct)],
        "Val acc (mean, min-max)": [f"{a * 100:.2f}% ({lo_ * 100:.2f}-{hi * 100:.2f})"
                                    for a, lo_, hi in zip(lo.acc_mean, lo.acc_min, lo.acc_max)],
        "Seeds": lo.n.tolist(),
    })
    if not full.empty:
        table.loc[len(table)] = ["full FT", "-", "-", "-",
                                 f"{int(full.trainable_params.iloc[0]):,} (100%)",
                                 f"{full.val_acc.mean() * 100:.2f}% ({full.val_acc.min() * 100:.2f}-"
                                 f"{full.val_acc.max() * 100:.2f})", len(full)]
    save_table(table, "h3_table")

    # two separate single-axis charts (never dual-axis)
    fig, ax = plt.subplots(figsize=(6.4, 4))
    x = list(lo.index)
    ax.plot(x, lo.acc_mean * 100, color=COLOR["lora"], lw=2, marker="o", ms=7, label="LoRA (mean of seeds)")
    ax.fill_between(x, lo.acc_min * 100, lo.acc_max * 100, color=COLOR["lora"], alpha=0.15, lw=0)
    if not full.empty:
        ax.axhline(full.val_acc.mean() * 100, color=COLOR["full"], ls="--", lw=2, label="Full fine-tune")
    ax.set_xscale("log", base=2); ax.set_xticks(x, [str(int(r)) for r in x])
    ax.set_xlabel("LoRA rank r"); ax.set_ylabel("SST-2 validation accuracy (%)")
    ax.set_title("H3 - accuracy vs rank (DistilBERT, SST-2)", loc="left"); ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(OUT / "h3_acc_vs_rank.png", dpi=200); plt.close(fig)

    fig, ax = plt.subplots(figsize=(6.4, 4))
    ax.plot(x, lo.lora, color=COLOR["lora"], lw=2, marker="o", ms=7)
    for r, v in zip(x, lo.lora):
        ax.annotate(fmt_params(v), (r, v), xytext=(0, 6), textcoords="offset points", ha="center", fontsize=9)
    ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_xticks(x, [str(int(r)) for r in x])
    ax.set_xlabel("LoRA rank r"); ax.set_ylabel("Trainable LoRA params (log)")
    ax.set_title("H3 - LoRA params grow linearly with r", loc="left")
    fig.tight_layout(); fig.savefig(OUT / "h3_params_vs_rank.png", dpi=200); plt.close(fig)
    return df


# ----------------------------------------------------------------------------- H4
def report_h4(h1: pd.DataFrame | None):
    df = load("h4")
    if df.empty:
        return print("H4: no results yet")
    ppl = df[df.status == "ok"].set_index("variant").wikitext2_ppl
    ref = {"full": "base", "lora": "base", "qlora": "base4bit"}
    held = {}
    if h1 is not None and not h1.empty:
        h = h1[h1.seed == df.h1_seed.iloc[0]].set_index("method")
        held = h.eval_loss_after.to_dict()
        # untrained references: the loss each base had before training (bf16 for full, 4-bit for qlora)
        if "full" in h.index:
            held["base"] = h.loc["full", "eval_loss_before"]
        if "qlora" in h.index:
            held["base4bit"] = h.loc["qlora", "eval_loss_before"]
    rows = []
    for v in ["base", "base4bit", "full", "lora", "qlora"]:
        if v not in ppl:
            continue
        r = ref.get(v)
        delta = f"{ppl[v] - ppl[r]:+.3f} ({100 * (ppl[v] / ppl[r] - 1):+.1f}%)" if r in ppl else "-"
        rows.append({"Model": {"base": "Base (bf16)", "base4bit": "Base (4-bit NF4)"}.get(v, LABEL.get(v, v)),
                     "WikiText-2 ppl (forgetting)": f"{ppl[v]:.3f}",
                     "Change vs its base": delta if r else "reference",
                     "Alpaca held-out loss (learning)": num(held.get(v), 3)})
    save_table(pd.DataFrame(rows), "h4_table")

    ms = [m for m in ORDER if m in ppl and ref[m] in ppl]
    if ms:
        rel = [100 * (ppl[m] / ppl[ref[m]] - 1) for m in ms]
        fig, ax = plt.subplots(figsize=(6.4, 4))
        bar(ax, [LABEL[m] for m in ms], rel, [COLOR[m] for m in ms], [f"{v:+.1f}%" for v in rel])
        ax.axhline(0, color=MUTED, lw=1)
        ax.set_ylabel("WikiText-2 perplexity change vs base (%)")
        ax.set_title("H4 - how much general text ability moved", loc="left")
        lo_, hi = min(0, min(rel)), max(0, max(rel))
        pad = (hi - lo_) * 0.2 or 1
        ax.set_ylim(lo_ - pad, hi + pad)
        fig.tight_layout(); fig.savefig(OUT / "h4_forgetting.png", dpi=200); plt.close(fig)


def footer(frames):
    frames = [f for f in frames if f is not None and not f.empty]
    if not frames:
        return
    d = pd.concat(frames, ignore_index=True)
    first = lambda c: d[c].dropna().iloc[0] if c in d and d[c].notna().any() else "?"  # noqa: E731
    jobs = ", ".join(sorted({str(j) for j in d.get("env_slurm_job_id", pd.Series()).dropna()}))
    txt = (f"Measured on Kaggle, 1x {first('env_gpu_name')} ({first('env_gpu_mem_gb')} GB), "
           f"PyTorch {first('env_torch')}, Transformers {first('env_transformers')}, PEFT {first('env_peft')}, "
           f"bitsandbytes {first('env_bitsandbytes')}, seed 42.")
    (OUT / "footer.txt").write_text(txt + "\n")
    print("\nSlide footer:\n" + txt)


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    h1 = report_h1()
    h2 = report_h2()
    h3 = report_h3()
    report_h4(h1)
    footer([h1, h2, h3])
    print(f"\nEverything written to {OUT}")


if __name__ == "__main__":
    main()


## H1 (needed for slide 4): full FT vs LoRA vs QLoRA on Qwen2.5-0.5B
Same settings as the HPC: 5,000 Alpaca examples (4,800 train, 200 held out), 300 steps = one epoch, seq len 512, batch 4×4, seed 42.

In [ ]:
AMP, STEPS = "fp16", 300
N_EXAMPLES = 5000   # 4,800 train + 200 held out: 300 steps x 16 = exactly one pass, no repeats (2,000 overfit full FT)
import os, json
for m in ["full", "lora", "qlora"]:
    f = f"results/h1/{m}_seed42.json"
    if os.path.exists(f) and json.load(open(f))["status"] == "ok":
        print("skip", m, "(already done)"); continue
    !python train_llm.py --exp h1 --method {m} --steps {STEPS} --n-examples {N_EXAMPLES} --amp {AMP}

In [ ]:
!python make_report.py
import os
from IPython.display import Markdown, Image, display
S = 'results/summary/'
if os.path.exists(S + 'h1_table.md'):
    display(Markdown(open(S + 'h1_table.md').read()))
for f in ['h1_memory.png', 'h1_speed.png']:
    if os.path.exists(S + f):
        display(Image(S + f))
for f in ['h1_ratios.txt', 'footer.txt']:
    if os.path.exists(S + f):
        print(open(S + f).read())

## Optional extras (run if time allows)
**H2**: Qwen2.5-3B on a 15 GB T4: full FT should OOM (recorded as a result), LoRA/QLoRA train 50 steps.  
**H3**: rank sweep, reduced for Colab time: r ∈ {1, 4, 16, 64}, one seed, 20k SST-2 examples, 1 epoch, plus a full-FT reference.  
**H4**: WikiText-2 perplexity of the H1 models vs the base (needs H1 above).

In [ ]:
# H2 (~15-20 min incl. 6 GB download)
for m in ["full", "lora", "qlora"]:
    !python train_llm.py --exp h2 --model Qwen/Qwen2.5-3B --method {m} --steps 50 --skip-eval-before --no-save --log-every 5 --amp {AMP}

In [ ]:
# H3 (~10-15 min)
for r in [1, 4, 16, 64]:
    !python train_h3.py --method lora --r {r} --seed 42 --train-subset 20000 --epochs 1 --amp {AMP}
!python train_h3.py --method full --seed 42 --train-subset 20000 --epochs 1 --amp {AMP}

In [ ]:
# H4 (~5 min)
!python eval_h4.py --variants base base4bit full lora qlora --amp {AMP}

## Build every table/chart and download

In [ ]:
!python make_report.py
import shutil
shutil.make_archive('/kaggle/working/results', 'zip', '.', 'results')
print('Done: download results.zip from the Output panel (or /kaggle/working)')